In [1]:
import pandas as pd
from pathlib import Path 
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
import numpy as np
from sklearn.preprocessing import TargetEncoder

# Data Cleaning and Preprocessing 

This notebook cleans and preprocessess historical real estate data used to develop a model for predicting home closing prices. 
## 1. Dataset Overview
The dataset consists of monthly CSV files containing historical real estate property records and their final sales prices (`ClosePrice`). The data is sourced from California Regional Multiple Listing Service (CRMLS) and covers January 2024 through April 2026.

Each CSV file represents one month of data. The monthly files are loaded and divided chronologically into training, validation, and test sets.

All preprocessing decisions and parameters are derived from the training data and then applied consistently to the validation and test sets to prevent data leakage.

In [2]:
# Create a Path object to the folder containing the monthly CSV files
data_path = Path("../data")

# Get all the CSV files from the data folder and sort them chronologically by filename
csv_files = sorted(data_path.glob("*.csv"))

### Train, Validation, and Test Split

The data is split chronologically rather than random to prevent future market conditions from leaking into the training .

- **Training set (Jan 2024 - Feb 2026):** All months except the two most recent complete months.
- **Validation set (Mar 2026):** The second-most-recent complete month, used for model selection and hyperparameter tuning.
- **Test set (April 2026):** The most recent complete month, reserved for final model evaluation.

In [3]:
# Split the monthly CSV files into training, validation, and test sets
train_files = csv_files[:-2]
validation_file = csv_files[-2]
test_file = csv_files[-1]

A `SourceMonth` column is added to each monthly training file based on its filename to identify the month and year of each record. The monthly files are then combined into a single DataFrame for preprocessing and modeling.  

In [4]:
# Read each monthly training file into a DataFrame
train_dfs = []

for file in train_files:
    monthly_data = pd.read_csv(file)

    # Extract the year and month from the CSV filename
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    monthly_data["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]
    
    train_dfs.append(monthly_data)
    
# Combine the monthly training DataFrames into a single DataFrame
initial_data = pd.concat(train_dfs, ignore_index=True)

/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1228/84281947.py:5: DtypeWarning: Columns (2,36,39,56,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1228/84281947.py:5: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)
/var/folders/s8/2hb9crtx0f59zm5tv6jsyvhm0000gn/T/ipykernel_1228/84281947.py:5: DtypeWarning: Columns (4,74) have mixed types. Specify dtype option on import or set low_memory=False.
  monthly_data = pd.read_csv(file)


In [5]:
# Load validation and test files
validation_df = pd.read_csv(validation_file)
test_df = pd.read_csv(test_file)

# Add source month to each dataset
for df, file in [(validation_df, validation_file), (test_df, test_file)]:
    source_month = file.stem.replace("CRMLSSold", "").replace("_filled", "")
    df["SourceMonth"] = source_month[:4] + "-" + source_month[4:6]

In [6]:
# Verify dataset sizes and source months
for name, df in [("Training", initial_data),
                 ("validation", validation_df),
                 ("test", test_df)]:

    print(f"{name}: {len(df):,} records")
    print(f"Months: {sorted(df['SourceMonth'].unique())}")
    print()

Training: 568,074 records
Months: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02']

validation: 23,372 records
Months: ['2026-03']

test: 24,261 records
Months: ['2026-04']



## 2. Initial Data Cleaning

In [7]:
def log_removed_rows(step, before, after):
    removed = before - after
    percentage = ((removed / before) * 100) if before > 0 else 0
    return {"step": step, "Before": before, "Removed": removed ,
            "Removed (%)": round(percentage, 4), "Remaining": after}

In [8]:
def initial_cleaning(df):
    # Create a copy of the original DateFrame
    df = df.copy()
    # Create an empty list to store a record of each cleaning step
    cleaning_log = []

    def log_step(step, before):
        # Add information about the cleaning step to the log
        cleaning_log.append(log_removed_rows(step, before, len(df)))
        
    # Filter CA records
    before = len(df)
    df = df[df["StateOrProvince"] == "CA"].copy()
    log_step("California filtering", before)

    # Convert dates
    for column in ["ListingContractDate", "CloseDate"]:
        df[column] = pd.to_datetime(df[column], errors="coerce")

    # Remove duplicate transactions using valid addresses
    before = len(df)
    invalid_addresses = ["0 0", "0 .", "N/A", "0 ,", "0 x", "E", "1 -", "0 --", "/", "0 I"]

    addresses = df["UnparsedAddress"].astype("string").str.strip()
    
    valid_address = (addresses.notna() &\
                ~addresses.isin(invalid_addresses) & addresses.ne(""))

    duplicates = (valid_address &\
                  df.assign(UnparsedAddress=addresses).duplicated(
                      subset=["UnparsedAddress", "CloseDate"], keep="first"))
    df = df.loc[~duplicates].copy()
    log_step("Duplicate removal", before)

    # Keep single-family residential properties
    df = df[(df["PropertyType"] == "Residential") &\
             (df["PropertySubType"] == "SingleFamilyResidence")].copy()

    log_step("Property-type filtering", before)

    # Remove inconsistent closing dates
    before = len(df)

    invalid_date_order = (df["CloseDate"] < df["ListingContractDate"])

    df = df.loc[~invalid_date_order].copy()
    log_step("Date consistency", before)
    

    # Remove missing or invalid target values
    before = len(df)

    df = df[df["ClosePrice"].notna() &
            (df["ClosePrice"] > 0)].copy()

    log_step("Traget cleaning", before)

    return df, pd.DataFrame(cleaning_log)
        

In [9]:
datasets = {
    "Training": initial_data, 
    "Validation": validation_df,
    "Test": test_df}

cleaned_datasets = {}
cleaning_logs = {}

for name, df in datasets.items():
    cleaned_datasets[name], cleaning_logs[name] = initial_cleaning(df)

clean_train_df = cleaned_datasets["Training"]
clean_validation_df = cleaned_datasets["Validation"]
clean_test_df = cleaned_datasets["Test"]

In [10]:
for name, log in cleaning_logs.items():
    print(f"\n{name} Cleaning Summary")
    display(log)


Training Cleaning Summary


,step,Before,Removed,Removed (%),Remaining
0,California filtering,568074,76,0.0134,567998
1,Duplicate removal,567998,2911,0.5125,565087
2,Property-type filtering,567998,283281,49.8736,284717
3,Date consistency,284717,39,0.0137,284678
4,Traget cleaning,284678,3,0.0011,284675



Validation Cleaning Summary


,step,Before,Removed,Removed (%),Remaining
0,California filtering,23372,1,0.0043,23371
1,Duplicate removal,23371,100,0.4279,23271
2,Property-type filtering,23371,11840,50.6611,11531
3,Date consistency,11531,1,0.0087,11530
4,Traget cleaning,11530,0,0.0000,11530



Test Cleaning Summary


,step,Before,Removed,Removed (%),Remaining
0,California filtering,24261,2,0.0082,24259
1,Duplicate removal,24259,101,0.4163,24158
2,Property-type filtering,24259,11802,48.6500,12457
3,Date consistency,12457,3,0.0241,12454
4,Traget cleaning,12454,0,0.0000,12454


## Cleaning Decisions

**State filtering:** Records with a `StateOrProvince` value other than `CA` are excluded because they fall outside the geographic scope of the prediction model. 

**Duplicate removal:** Transactions with the same valid address and closing date are treated as duplicates, retaining the first occurrence. Missing or invalid addresses are excluded from duplicate matching.

**Property-type filtering:** Only records classified as `Residential` and `SingleFamilyResidence` are retained.

**Date consistency:** Records where `CloseDate` precedes `ListingContractDate` are removed because the reported transaction chronology is inconsistent.

**Target cleaning:** Records with missing, zero, or negative `ClosePrice` values are removed because they do not provide valid training or evaluation targets

## Feature Selection

Candidate features are selected based on whether they would be available at the time property is valued, before any sale process or outcome is known. Features associated with the listing or sale process, such as `ListPrice`, `DaysOnMarket`, and `OriginalPrice` are excluded from the model to prevent data leakage. Property characteristics that exist independently of the sale process, such as living area, bedrooms, bathrooms, lot size, year built, and location are retained as candidate predictors. `ClosePrice` is retained only as the target variable and is not included as an input feature.

Note: `SourceMonth` is kept for tracking rather than modeling.

In [11]:
feature_columns = ["AttachedGarageYN", "BasementYN", "BathroomsTotalInteger", "BedroomsTotal",
               "BuildingAreaTotal", "City", "CountyOrParish", "FireplaceYN",
               "Flooring", "GarageSpaces", "Latitude", "Longitude", "LivingArea", "Levels",
               "LotSizeSquareFeet", "LotSizeAcres", "NewConstructionYN", "ParkingTotal",
               "PoolPrivateYN", "PostalCode", "Stories", "ViewYN", "YearBuilt"]

target_column = ["ClosePrice"]
tracking_column = ["SourceMonth"]

key_columns = feature_columns + target_column + tracking_column

In [12]:
clean_train_df = clean_train_df[key_columns].copy()
clean_validation_df = clean_validation_df[key_columns].copy()
clean_test_df = clean_test_df[key_columns].copy()

In [13]:
for name, df in [
    ("Training", clean_train_df), 
    ("Validation", clean_validation_df),
    ("Test", clean_test_df)]:
    print(f"{name}: {df.shape}")

Training: (284675, 25)
Validation: (11530, 25)
Test: (12454, 25)


In [14]:
# Perserve the datasets before further preprocessing
base_train_df = clean_train_df.copy()
base_validation_df = clean_validation_df.copy()
base_test_df = clean_test_df.copy()

### Training Window Selection

The training window (X) is a tunable parameter representing the number of consecutive months used for model training immediately before the validation period (March 2026).

The full 26-month training dataset is preserved, allowing different training windows to be evaluated without repeating initial cleaning. The selected training window is used to determine missingness thresholds and fit preprocessing parameters, which are subsequently applied to validation and test datasets. The optimal training window will be evaluated during model development.

In [15]:
# Tunable training window
X = 26 # change to 6, 12, etc. for further experiments 

# Reset datasets from preserved copies
clean_train_df = base_train_df.copy()
clean_validation_df = base_validation_df.copy()
clean_test_df = base_test_df.copy()

# Convert SourceMonth to monthly periods
months = pd.PeriodIndex(clean_train_df["SourceMonth"], freq="M")

# March 2026 is the month used for validation dataset
validation_month = pd.Period("2026-03", freq="M")

# Select the X months immediately before validation
clean_train_df = clean_train_df.loc[(months >= validation_month - X) & (months < validation_month)].copy()

print(f"Training window: {X} months")
print(f"Training period: {validation_month - X} to {validation_month - 1}")
print(f"Training records: {len(clean_train_df)}")

Training window: 26 months
Training period: 2024-01 to 2026-02
Training records: 284675


### ClosePrice Outlier Filtering

The 0.5th and 99.5th percentiles of `ClosePrice` are calculated using the selected training window only. These fixed thresholds are applied consistently to the training, validation, and test datasets to remove extreme sale prices without introducing data leakage.

In [16]:
def filter_close_price(df, lower_threshold, upper_threshold):
    df = df.copy()

    before = len(df)

    df = df[df["ClosePrice"].between(lower_threshold, upper_threshold)].copy()
    log = log_removed_rows("ClosPrice outlier filtering", before, len(df))
    
    return df, log

In [17]:
# Calculate thresholds from the selected X-month training window
lower_threshold = clean_train_df["ClosePrice"].quantile(0.005)
upper_threshold = clean_train_df["ClosePrice"].quantile(0.995)

print(f"Lower ClosePrice threshold: ${lower_threshold}")
print(f"Upper ClosePrice threshold: ${upper_threshold}")

Lower ClosePrice threshold: $190000.0
Upper ClosePrice threshold: $8300000.0


In [18]:
datasets = {
    "Training": clean_train_df, 
    "Validation": clean_validation_df,
    "Test": clean_test_df}

# Initialize containers for cleaned datasets and removal logs
cleaned_datasets = {}
close_price_logs = []

# Apply LivingArea cleaning to each dataset
for name, df in datasets.items():
    # Remove records with zero and negative LivingArea values and return the cleaning log
    cleaned_df, log = filter_close_price(df, lower_threshold, upper_threshold)

    cleaned_datasets[name] = cleaned_df
    log["Dataset"] = name
    close_price_logs.append(log)

# update datasets
clean_train_df = cleaned_datasets["Training"]
clean_validation_df = cleaned_datasets["Validation"]
clean_test_df = cleaned_datasets["Test"]

display(pd.DataFrame(close_price_logs))

,step,Before,Removed,Removed (%),Remaining,Dataset
0,ClosPrice outlier filtering,284675,2813,0.9881,281862,Training
1,ClosPrice outlier filtering,11530,139,1.2056,11391,Validation
2,ClosPrice outlier filtering,12454,139,1.1161,12315,Test


### ClosePrice Log Transformation

Following outlier filtering, a natural logarithm transformation is applied to `ClosePrice` to reduce right-skewness. The resulting `LogClosePrice` is used as the prediction target and is created consistently for training, validation, and test datasets. 

The original `ClosePrice` is retained for evaluation in dollar units.

In [19]:
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df["LogClosePrice"] = np.log(df["ClosePrice"])

### Missing Values

Missing values are examined across the candidate features to understand the extent of missingness before making data preprocessing decisions.

In [20]:
# Identify features with at least 90% missing values
missing_percent = clean_train_df[feature_columns].isna().mean().mul(100).sort_values(ascending=False)
missing_percent

BasementYN               97.671556
BuildingAreaTotal        93.825702
Flooring                 35.600755
Stories                  12.786399
AttachedGarageYN         11.257282
PoolPrivateYN             9.679560
Levels                    9.600443
ViewYN                    9.000149
NewConstructionYN         7.271644
GarageSpaces              3.625178
LotSizeAcres              1.729570
LotSizeSquareFeet         1.719636
Longitude                 0.900795
Latitude                  0.900795
ParkingTotal              0.123110
City                      0.080181
YearBuilt                 0.060668
FireplaceYN               0.057830
LivingArea                0.041510
BathroomsTotalInteger     0.017030
PostalCode                0.000355
CountyOrParish            0.000000
BedroomsTotal             0.000000
dtype: float64

### Missing Value Threshold

Features with 90% or more missing values are excluded from modeling. Based on the full 26 months training dataset, `BasementYN` and `BuildingAreaTotal` exceed this threshold. 

Since missingness percentages may vary depending on the selected training window (X months), the threshold will be reevaluated for each training window. The resulting feature selection will be applied consistently to the training, validation, and test datasets to prevent data leakage.


In [21]:
# Identify features with at least 90% missing values
missing_threshold = 90

columns_to_drop = missing_percent[missing_percent >= missing_threshold].index.tolist()
print(F"Columns to drop: {columns_to_drop}")

# Apply the same column removal to all datasets
for df in [clean_train_df, clean_validation_df, clean_test_df]:
    df.drop(columns=columns_to_drop, inplace=True)

Columns to drop: ['BasementYN', 'BuildingAreaTotal']


### Living Area Validation

Records with zero or negative `LivingArea` are removed because these values are logically invalid for residential properties. Missing `LivingArea` values are retained and subsequently imputed using the median calculated from the selected training window. 

In [22]:
def clean_living_area(df):
    df = df.copy()

    df["LivingArea"] = pd.to_numeric(df["LivingArea"], errors="coerce")
    before = len(df)

    # Remove zero and negative LivingArea values, but retain missing values
    df = df[df["LivingArea"].isna() | (df["LivingArea"] > 0)].copy()

    log = log_removed_rows("Zero and negative LivingArea", before, len(df))

    return df, log

In [23]:
datasets = {
    "Training": clean_train_df, 
    "Validation": clean_validation_df,
    "Test": clean_test_df}

# Initialize containers for cleaned datasets and removal logs
cleaned_datasets = {}
living_area_logs = []

# Apply LivingArea cleaning to each dataset
for name, df in datasets.items():
    # Remove records with zero and negative LivingArea values and return the cleaning log
    cleaned_df, log = clean_living_area(df)

    cleaned_datasets[name] = cleaned_df
    log["Dataset"] = name
    living_area_logs.append(log)

# update datasets
clean_train_df = cleaned_datasets["Training"]
clean_validation_df = cleaned_datasets["Validation"]
clean_test_df = cleaned_datasets["Test"]

display(pd.DataFrame(living_area_logs))

,step,Before,Removed,Removed (%),Remaining,Dataset
0,Zero and negative LivingArea,281862,100,0.0355,281762,Training
1,Zero and negative LivingArea,11391,1,0.0088,11390,Validation
2,Zero and negative LivingArea,12315,5,0.0406,12310,Test


### LivingArea Imputation and Log Transformation

After removing invalid values, `100` missing values remain in training, 1 in validation, and 5 in test. These are imputed using the training median of `1,801` sq ft. 

The LogLivingArea feature is then created using the natural logarithm to reduce right skew.

In [24]:
# Calculate the median using the selected X-month training window
living_area_median = clean_train_df["LivingArea"].median()
print(f"Training LivingArea median: {living_area_median:.2f}")

Training LivingArea median: 1801.00


In [25]:
# Apply the same median andlog transformation to all three datasets
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df["LivingArea"] = df["LivingArea"].fillna(living_area_median)
    df["LogLivingarea"] = np.log(df["LivingArea"])

### Lot Size 

`LotSizeSquareFeet` and `LotSizeAcres` were examined for consistency. Approximately `99.89%` of records with valid values in both columns agreed within 1%, while 308 records showed larger discrepancies. 

In [26]:
# Check consistency between lot-size measurements
lot_size_sqft = pd.to_numeric(clean_train_df["LotSizeSquareFeet"], errors="coerce")
lot_size_acres = pd.to_numeric(clean_train_df["LotSizeAcres"], errors="coerce")

# Compare records with positive values in both columns
valid = (lot_size_sqft > 0) & (lot_size_acres > 0)

# Calculate the relative difference
difference = (lot_size_sqft[valid] - lot_size_acres[valid] * 43560).abs() / lot_size_sqft[valid]

# Report consistency
consistent = (difference <= 0.01).sum()
inconsistent = (difference > 0.01).sum()

print(f"Records agreeing within 1%: {consistent / valid.sum() * 100:.2f}%")
print(f"Records disagreeing by more than 1%: {inconsistent}")

Records agreeing within 1%: 99.89%
Records disagreeing by more than 1%: 307


### Lot Size Validation and Imputation

Zero and negative values are treated as missing. Missing `LotSizeSquareFeet` values are recovered using `LotSizeAcres` (1 acre = 43,560 sq ft), while inconsistent measurments remain unchanged. Remaining missing values are imputed using the training dataset median of `LotSizeSquareFeet` `(7,223 sq ft)` and `LotSizeAcres` is dropped to reduce redundancy.

In [27]:
def clean_lot_size(df):
    df = df.copy()

    # Convert both lot-size columns to numeric
    lot_columns = ["LotSizeSquareFeet", "LotSizeAcres"]

    for column in lot_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

        # Treat zeros and negative values as missing
        df.loc[df[column] <= 0, column] = np.nan

    # Identify records where square footage is missing but acres is valid
    missing_lotsize_sqft = (df["LotSizeSquareFeet"].isna() & df["LotSizeAcres"].notna())

    # Recover missing square footage using acres
    recovered_rows = missing_lotsize_sqft.sum()
    df.loc[missing_lotsize_sqft , "LotSizeSquareFeet"] = ( 
           df.loc[missing_lotsize_sqft, "LotSizeAcres"] * 43560)

    # Drop the redundant acreage feature
    df = df.drop(columns=["LotSizeAcres"])

    return df, recovered_rows

In [28]:
datasets = {
    "Training": clean_train_df, 
    "Validation": clean_validation_df,
    "Test": clean_test_df}

# Initialize containers for cleaned datasets and removal logs
cleaned_datasets = {}
lot_size_logs = []

# Apply the same lot-size cleaning rules to each dataset
for name, df in datasets.items():
    cleaned_df, recovered_rows = clean_lot_size(df)

    cleaned_datasets[name] = cleaned_df
    log["Dataset"] = name
    lot_size_logs.append({"Dataset": name, "Recovered from Arces": recovered_rows,
                          "Remaining Missing": cleaned_df["LotSizeSquareFeet"].isna().sum()})
    
# Update datasets 
clean_train_df = cleaned_datasets["Training"]
clean_validation_df = cleaned_datasets["Validation"]
clean_test_df = cleaned_datasets["Test"]

display(pd.DataFrame(lot_size_logs))

,Dataset,Recovered from Arces,Remaining Missing
0,Training,18,4971
1,Validation,2,188
2,Test,0,249


In [29]:
# Calculate the median using the selected X-month training window
lot_size_median = clean_train_df["LotSizeSquareFeet"].median()
print(f"Training LotSizeSquareFeet median: {lot_size_median:.2f}")

Training LotSizeSquareFeet median: 7223.00


In [30]:
# Apply the same median andlog transformation to all three datasets
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df["LotSizeSquareFeet"] = df["LotSizeSquareFeet"].fillna(lot_size_median)

### Property Characteristics Validation

Numerical property characteristics are validated using provisional plausibility thresholds informed by the training data. Implausible values are replaced with missing values rather than corrected without supporting evidence. For example, unusually large values in BathroomsTotalInteger cannot be reliably reconstructed because separate bathroom-component fields are unavailable.

The validation thresholds will be reviewed with the team during the upcoming meeting and may be revised based on team agreement and further analysis. Any revisions will be documented and applied consistently across the training, validation, and test datasets.

In [31]:
def clean_property_characteristics(df):
    df = df.copy()
    
    property_columns = ["BedroomsTotal", "BathroomsTotalInteger","GarageSpaces",
    "ParkingTotal", "Stories", "YearBuilt"]

    # Convert property characteristics to numeric
    for column in property_columns:
        df[column] = pd.to_numeric(df[column], errors="coerce")

    # Define valid ranges for property counts
    valid_ranges = {"BedroomsTotal": (0, 20),
                    "BathroomsTotalInteger": (0, 20),
                    "GarageSpaces": (0, 20),
                    "ParkingTotal": (0, 100),
                    "Stories": (1, None)}
    invalid_logs = {}

    # Replace invalid property counts with NAN
    for column, (lower, upper) in valid_ranges.items():
        # Only check non-missing values
        invalid = df[column].notna() &\
                ((df[column] < lower) | (df[column] % 1 != 0))

        if upper is not None:
            invalid |= df[column] > upper

        invalid_logs[column] = int(invalid.sum())
        df.loc[invalid, column] = np.nan

    invalid_year = ((df["YearBuilt"] <= 0) |\
                   (df["YearBuilt"] % 1 != 0))
    
    invalid_logs["YearBuilt"] = int(invalid_year.sum())
    df.loc[invalid_year, "YearBuilt"] = np.nan

    return df, invalid_logs

In [32]:
# Apply property characteristics validation to all datasets
clean_train_df, _ = clean_property_characteristics(clean_train_df)
clean_validation_df, _ = clean_property_characteristics(clean_validation_df)
clean_test_df, _ = clean_property_characteristics(clean_test_df)

In [33]:
mode_columns = ["BedroomsTotal", "BathroomsTotalInteger", "GarageSpaces",
                "ParkingTotal", "Stories"]

# Calculate imputation values from training only
property_modes = clean_train_df[mode_columns].mode().iloc[0]
year_built_median = round(clean_train_df["YearBuilt"].median())

# Record and impute missing values
imputation_logs = []

for name, df in {"Training": clean_train_df, "Validation": clean_validation_df,
                 "Test": clean_test_df}.items():

    for col in mode_columns:
        missing_count = int(df[col].isna().sum())
        df[col] = df[col].fillna(property_modes[col])

        imputation_logs.append({"Dataset": name, "Feature": col,
                                "Imputed Values": missing_count, "Imputation Value": property_modes[col]})

    missing_count = int(df["YearBuilt"].isna().sum())
    df["YearBuilt"] = df["YearBuilt"].fillna(year_built_median)

    imputation_logs.append({"Dataset": name, "Feature": "YearBuilt",
                            "Imputed Values": missing_count, "Imputation Value": year_built_median})

display(pd.DataFrame(imputation_logs))

,Dataset,Feature,Imputed Values,Imputation Value
0,Training,BedroomsTotal,4,3.0
1,Training,BathroomsTotalInteger,55,2.0
2,Training,GarageSpaces,10689,2.0
3,Training,ParkingTotal,1034,2.0
4,Training,Stories,36029,1.0
5,Training,YearBuilt,149,1976.0
6,Validation,BedroomsTotal,0,3.0
7,Validation,BathroomsTotalInteger,1,2.0
8,Validation,GarageSpaces,419,2.0
9,Validation,ParkingTotal,24,2.0


### Postal Code Standardization

The `PostalCode` feature was standardized to a five-digit string format and applied to the training, validation, and test datasets to ensure consistency before geographic coordinate imputation.

In [34]:
def clean_postal_code(df):
    df = df.copy()

    # Convert postal codes to strings and remove whitespace
    postal = df["PostalCode"].astype("string").str.strip()

    # Extract the first five digits from ZIP+4 codes
    postal = postal.str.replace(r"^(\d{5})-\d{4}$", r"\1", regex=True)

    # Keep only valid five-digit formats
    df["PostalCode"] = postal.where(postal.str.fullmatch(r"\d{5}"))

    return df

In [35]:
clean_train_df = clean_postal_code(clean_train_df)
clean_validation_df = clean_postal_code(clean_validation_df)
clean_test_df = clean_postal_code(clean_test_df)

### Geographic Coordinate Validation and Imputation

Latitude and longitude values outside approximate California geographic bounds were treated as missing. Missing coordinates were imputed using median coordinates grouped by `PostalCode`, calculated exclusively from the selected training window and applied consistently to training, validation, and test datasets.

Following imputation, 10 training records and 1 validation record still had missing coordinates. These records were removed due to their small number. No test records were removed during this final step.

In [36]:
def clean_coordinates(df):
    df = df.copy()
    
    # Identify latitude values outside CA bounds
    invalid_latitude = (
        df["Latitude"].notna() &
        ~df["Latitude"].between(32.5, 42.1)
    )

    # Identify longitude values outside CA bounds
    invalid_longitude = (
        df["Longitude"].notna() &
        ~df["Longitude"].between(-124.5, -114.0)
    )

    # Mark both coordinates missing if either is invalid
    invalid_pair = invalid_latitude | invalid_longitude

    df.loc[invalid_pair, ["Latitude", "Longitude"]] = np.nan

    return df

In [37]:
# Validate geographic coordinates across all datasets
clean_train_df = clean_coordinates(clean_train_df)
clean_validation_df = clean_coordinates(clean_validation_df)
clean_test_df = clean_coordinates(clean_test_df)

In [38]:
# Calculate median coordinates by ZIP code using training only
zip_coordinates = (clean_train_df.\
    groupby("PostalCode")[["Latitude", "Longitude"]].median())

In [39]:
def impute_coordinates(df, zip_coordinates):
    df = df.copy()

    missing_before = df["Latitude"].isna() | df["Longitude"].isna()

    # Look up training-derived median coordinates
    for column in ["Latitude", "Longitude"]:
        zip_medians = df["PostalCode"].map(zip_coordinates[column])
        df[column] = df[column].fillna(zip_medians)

    missing_after = df["Latitude"].isna() | df["Longitude"].isna()

    recovered = int(missing_before.sum() - missing_after.sum())

    return df, recovered

In [40]:
clean_train_df, recovered_train = impute_coordinates(clean_train_df, zip_coordinates)
clean_validation_df, recovered_val = impute_coordinates(clean_validation_df, zip_coordinates)
clean_test_df, recovered_test = impute_coordinates(clean_test_df, zip_coordinates)

In [41]:
datasets = {
    "Training": clean_train_df,
    "Validation": clean_validation_df,
    "Test": clean_test_df
}

geo_logs = []

for name, df in datasets.items():
    before = len(df)

    # Remove records with missing latitude or longitude
    datasets[name] = df.dropna(subset=["Latitude", "Longitude"]).copy()

    log = log_removed_rows("Missing coordinates", before, len(datasets[name]))
    log["Dataset"] = name
    geo_logs.append(log)

# Update datasets
clean_train_df = datasets["Training"]
clean_validation_df = datasets["Validation"]
clean_test_df = datasets["Test"]

display(pd.DataFrame(geo_logs))

,step,Before,Removed,Removed (%),Remaining,Dataset
0,Missing coordinates,281762,10,0.0035,281752,Training
1,Missing coordinates,11390,1,0.0088,11389,Validation
2,Missing coordinates,12310,0,0.0000,12310,Test


### Binary Encoding and Imputation

Five boolen property features were encoded as binary variables with True mapped to 1 and False mapped to 0. Missing values were imputed using mode calculated from the selectedtraining window. The same encoding and training-derived imputation values were applied consistently to the training, validation, and test dataset to prevent data leakage.

In [42]:
binary_columns = ["AttachedGarageYN", "FireplaceYN", "NewConstructionYN", "PoolPrivateYN", "ViewYN"] 

In [43]:
def encode_binary_features(df):
    df = df.copy()

    # Convert True to 1 and False to 0
    for column in binary_columns:
        df[column] = df[column].map({True: 1, False: 0})

    return df

In [44]:
# Apply validation to all datasets
clean_train_df = encode_binary_features(clean_train_df)
clean_validation_df = encode_binary_features(clean_validation_df)
clean_test_df = encode_binary_features(clean_test_df)

In [45]:
# Calculate modes using training data only
binary_modes = clean_train_df[binary_columns].mode().iloc[0]

for df in[clean_train_df, clean_validation_df, clean_test_df]:
    df[binary_columns] = df[binary_columns].fillna(binary_modes)

print(binary_modes)

AttachedGarageYN     1.0
FireplaceYN          1.0
NewConstructionYN    0.0
PoolPrivateYN        0.0
ViewYN               1.0
Name: 0, dtype: float64


### Categorical Target Encoding

The high-cardinality categorical features `City`, `CountyOrParish`, and `PostalCode` were encoded using random five-fold cross-fitted target encoding with `LogClosePrice` as the target. Cross-fitting prevents each training observation from using its own target value when calculating its encoded features.

The encoder was fitted using the training dataset only, and the learned mappings were applied to the validation and test datasets to prevent target leakage. Three numerical features were created: `City_encoded`, `CountyOrParish_encoded`, and `PostalCode_encoded`.

In [46]:
target_encode_cols = ["City", "CountyOrParish", "PostalCode"]

In [47]:
# Standarize categories and replace missing values with "Unknown"
for df in[clean_train_df, clean_validation_df, clean_test_df]:
    for column in target_encode_cols:
        df[column] = (df[column].astype(str).fillna("Unknown"))

In [48]:
# Initialize target encoder with 5-fold cross fitting
encoder = TargetEncoder(target_type="continuous", cv=5, shuffle=True, random_state=42)

In [49]:
# Encode training features using cross fitting to prevent target leakage
X_train_encoded = encoder.fit_transform(clean_train_df[target_encode_cols], clean_train_df["LogClosePrice"])

In [50]:
# Apply training-fitted encoding to validation and test datasets
X_val_encoded = encoder.transform(clean_validation_df[target_encode_cols])
X_test_encoded = encoder.transform(clean_test_df[target_encode_cols])

In [51]:
# Define names for the new encoded features
encoded_columns = ["City_encoded", "CountyOrParish_encoded", "PostalCode_encoded"]

In [52]:
# Add encoded features to the corresponding datasets
clean_train_df[encoded_columns] = X_train_encoded
clean_validation_df[encoded_columns] = X_val_encoded
clean_test_df[encoded_columns] = X_test_encoded

In [53]:
# Print dataset sizes before and after cleaning 
for name, df in [
    ("Training before cleaning", initial_data), ("Training after cleaning", clean_train_df),
    ("Validation before cleaning", validation_df), ("Validation after cleaning", clean_validation_df),
    ("Test before cleaning", test_df), ("Test after cleaning", clean_test_df)]:

    print(f"{name}: {len(df):,} records")

Training before cleaning: 568,074 records
Training after cleaning: 281,752 records
Validation before cleaning: 23,372 records
Validation after cleaning: 11,389 records
Test before cleaning: 24,261 records
Test after cleaning: 12,310 records


In [54]:
output_dir = Path("../cleaned_data")
output_dir.mkdir(parents=True, exist_ok=True)

clean_train_df.to_csv(output_dir/"train_clean.csv", index=False)
clean_validation_df.to_csv(output_dir/"validation_clean.csv", index=False)
clean_test_df.to_csv(output_dir/"test_clean.csv", index=False)